# Inspect Mask Values

Label mask tifs (values 0/1/2) look black/near-black in a normal viewer because
those values sit at the very bottom of an 8-bit 0-255 range — that alone isn't
evidence the masks are empty. This notebook prints the actual unique pixel
values per file and renders a proper colour-mapped preview so you can eyeball
whether corrections actually landed.


In [ ]:
from pathlib import Path

import numpy as np
import tifffile
import matplotlib.pyplot as plt
from skimage.color import label2rgb


## Config

In [ ]:
MASK_DIR = Path("/path/to/your/masks_corrected")
PATTERN = "*.tif"
PREVIEW_N = 6  # how many masks to show in the colour preview grid

files = sorted(MASK_DIR.glob(PATTERN))
print(f"Found {len(files)} file(s) matching {PATTERN} in {MASK_DIR}")
assert files, "No files found — check MASK_DIR / PATTERN."


## Per-file value breakdown

In [ ]:
def summarize(path):
    arr = tifffile.imread(path)
    values, counts = np.unique(arr, return_counts=True)
    total = arr.size
    breakdown = ", ".join(
        f"{v}: {c} ({100 * c / total:.1f}%)" for v, c in zip(values, counts)
    )
    print(f"{path.name}")
    print(f"    shape={arr.shape}  dtype={arr.dtype}")
    print(f"    unique values -> {breakdown}")
    return values

all_values = set()
for f in files:
    values = summarize(f)
    all_values.update(values.tolist())
    print()

print(f"Value range across all files: {sorted(all_values)}")
if all_values == {0}:
    print("\n*** All masks contain only 0 — these really are empty/uncorrected. ***")
else:
    print("\nNon-zero labels are present — the masks are not actually empty, "
          "they just look black at raw 0-255 display scaling.")


## Colour-mapped preview

Same masks, rendered through `label2rgb` so 0/1/2 map to visually distinct
colours instead of near-black grey levels.


In [ ]:
n = min(PREVIEW_N, len(files))
preview_files = files[:n]

fig, axes = plt.subplots(1, n, figsize=(3 * n, 3))
if n == 1:
    axes = [axes]

for ax, f in zip(axes, preview_files):
    arr = tifffile.imread(f)
    rgb = label2rgb(arr.astype(np.int32), bg_label=-1)
    ax.imshow(rgb)
    ax.set_title(f.name, fontsize=8)
    ax.axis("off")

plt.tight_layout()
plt.show()


In [ ]:
# Optional: save the preview to disk
out_path = MASK_DIR / "_value_preview.png"
fig.savefig(out_path, dpi=150)
print(f"Saved preview -> {out_path}")
